In [1]:
# SCRIPT 2 PRO – ENSEMBLE + CALIBRACIÓN EN VALIDACIÓN + RECALL-FIRST THRESHOLD
# Meta: Recall ≳ 70–75% y Precision ≥ 5%, aceptando más cómputo.

import warnings, os, gc, time, joblib, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    precision_recall_curve, average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV

try:
    from lightgbm import LGBMClassifier
    import lightgbm as lgb
    print("✅ LightGBM disponible")
except ImportError:
    print("❌ LightGBM no disponible")
    raise SystemExit(1)

✅ LightGBM disponible


In [2]:
print("🚀 SCRIPT 2 PRO: ENSEMBLE + CALIBRACIÓN VALID + RECALL-FIRST")
print("=" * 80)

# ==========================
# CONFIG
# ==========================
INPUT_DIR   = "../data/processed_ml"
OUTPUT_DIR  = "../models"
RESULTS_DIR = "../results"
DATA_PATH   = "../data/processed/df_ready_model.csv"  # para históricos

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE   = 42
TARGET_RECALL  = 0.75   # objetivo primario
MIN_PRECISION  = 0.05   # mínimo deseado
RECALL_TOL     = 0.02   # tolerancia de recall (p.ej. 0.73–0.75)
MAX_POS_RATE   = 0.25   # cap de % de positivos en valid para evitar avalanchas de FP
TOP_K_ENSEMBLE = 3      # nº de mejores configs para el ensamblado

def logp(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

start_time = time.time()

# ==========================
# 1) CARGA DE DATOS
# ==========================
logp("Cargando datos procesados...")
path_bundle = os.path.join(INPUT_DIR, "training_data_ready.joblib")
if not os.path.exists(path_bundle):
    print(f"❌ No existe {path_bundle}. Ejecuta el script 1 primero.")
    raise SystemExit(1)

bundle = joblib.load(path_bundle)

X_train, y_train = bundle['X_train'], bundle['y_train']
X_val,   y_val   = bundle['X_val'],   bundle['y_val']
X_test,  y_test  = bundle['X_test'],  bundle['y_test']

scaler            = bundle.get('scaler')
selector          = bundle.get('selector')
selected_features = bundle.get('selected_features')
feature_cols      = bundle.get('feature_cols')
cluster_geometries= bundle.get('cluster_geometries')
metadata          = bundle.get('metadata', {})

logp(f"Tamaños -> Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")
logp(f"Tasa accidente -> Train: {y_train.mean()*100:.2f}% | Val: {y_val.mean()*100:.2f}%")

# Dataset original para históricos
df_original = None
try:
    logp("Cargando dataset original para históricos…")
    df_original = pd.read_csv(
        DATA_PATH,
        dtype={
            'accident':'int8','year':'int16','month':'int8','day':'int8',
            'hour':'int8','day_of_week':'int8','cluster_id':'int16'
        }
    )
    logp(f"Original: {len(df_original):,} filas")
except FileNotFoundError:
    logp("⚠️ No se encontró el dataset original, históricos extendidos desactivados.")

# ==========================
# 2) RANDOMIZED SEARCH AMPLIADO (AP)
# ==========================
logp("RandomizedSearch ampliado (scoring=average_precision)…")

# Nota: mantenemos class_weight='balanced' y, además, probamos scale_pos_weight
# en torno a 1 para ajustar el empuje hacia positivos (afecta a precisión/recall).
param_distributions = {
    'n_estimators':      [300, 400, 500, 700, 900, 1100],
    'learning_rate':     [0.01, 0.02, 0.03, 0.05],
    'num_leaves':        [15, 31, 50, 63, 80, 127],
    'min_child_samples': [50, 80, 100, 150, 200],
    'max_depth':         [-1, 8, 10, 12],
    'reg_alpha':         [0.0, 0.5, 1.0, 2.0, 5.0],
    'reg_lambda':        [0.5, 1.0, 2.0, 5.0, 10.0],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.7, 0.8, 0.9, 1.0],
    'min_split_gain':    [0.0, 0.1, 0.5, 1.0],
    'scale_pos_weight':  [0.8, 1.0, 1.2, 1.5]   # <1 empuja a más precisión, >1 a más recall
}

base_model = LGBMClassifier(
    objective='binary',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    importance_type='gain'
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=48,                 # ↑ para buscar mejor compromiso
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=False
)

rs.fit(X_train, y_train)
logp(f"✅ Mejor AP (CV): {rs.best_score_:.4f}")
cvres = pd.DataFrame(rs.cv_results_).sort_values('mean_test_score', ascending=False).reset_index(drop=True)

# ==========================
# 3) ENSEMBLE (TOP-K) + CALIBRACIÓN EN VALIDACIÓN
# ==========================
logp(f"Entrenando ensemble top-{TOP_K_ENSEMBLE} + calibración en validación…")

models = []
calibrators = []
val_probas_list = []
test_probas_list = []

for i in range(min(TOP_K_ENSEMBLE, len(cvres))):
    params = {k.replace('param_', ''): cvres.loc[i, k] for k in cvres.columns if k.startswith('param_')}
    params = {k: (None if v is None else v) for k, v in params.items()}  # limpiar
    
    m = LGBMClassifier(
        **params,
        objective='binary',
        class_weight='balanced',
        random_state=RANDOM_STATE + i,
        n_jobs=-1,
        verbose=-1
    )
    # early stopping contra validación (temporalmente coherente)
    m.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='binary_logloss',
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0)
        ]
    )
    # Calibración isotónica **en validación** (para alinear bien con test)
    cal = CalibratedClassifierCV(m, method='isotonic', cv='prefit')
    cal.fit(X_val, y_val)
    
    val_probas = cal.predict_proba(X_val)[:, 1]
    test_probas = cal.predict_proba(X_test)[:, 1]
    
    models.append(m)
    calibrators.append(cal)
    val_probas_list.append(val_probas)
    test_probas_list.append(test_probas)

# Probabilidades promediadas (bagging simple)
proba_val_ens  = np.mean(np.column_stack(val_probas_list),  axis=1)
proba_test_ens = np.mean(np.column_stack(test_probas_list), axis=1)

# ==========================
# 4) SELECCIÓN DE UMBRAL (RECALL-FIRST CON RESTRICCIONES)
# ==========================
logp("Buscando umbral (recall-first, precisión mínima, cap de positivos)…")

# ordenamos por proba para cálculo acumulado estable
order    = np.argsort(-proba_val_ens)
p_sorted = proba_val_ens[order]
y_sorted = y_val[order]

tp_cum   = np.cumsum(y_sorted)
k        = np.arange(1, len(y_sorted) + 1)
pos_tot  = y_val.sum()
n_val    = len(y_val)

rec_k       = tp_cum / max(pos_tot, 1)
prec_k      = tp_cum / k
pos_rate_k  = k / n_val
thr_k       = p_sorted - 1e-12

def f05(p, r):
    b2 = 0.5**2
    return (1 + b2) * (p * r) / (b2 * p + r) if (p + r) > 0 else 0.0

# candidatos “estrictos”
mask_ok = (
    (rec_k >= (TARGET_RECALL - RECALL_TOL)) &
    (prec_k >= MIN_PRECISION) &
    (pos_rate_k <= MAX_POS_RATE)
)
idx_ok = np.where(mask_ok)[0]

if len(idx_ok) > 0:
    scores = np.array([f05(prec_k[i], rec_k[i]) for i in idx_ok])
    best_idx = idx_ok[np.lexsort((-prec_k[idx_ok], -scores))][0]
else:
    # “suaves” (por si los estrictos no existen)
    mask_soft = (
        (rec_k >= 0.70) & (prec_k >= 0.035) & (pos_rate_k <= min(0.35, MAX_POS_RATE + 0.10))
    )
    idx_soft = np.where(mask_soft)[0]
    if len(idx_soft) > 0:
        scores = np.array([f05(prec_k[i], rec_k[i]) for i in idx_soft])
        best_idx = idx_soft[np.lexsort((-prec_k[idx_soft], -scores))][0]
    else:
        # último recurso: controlar tasa de positivos por percentil
        cutoff = np.percentile(proba_val_ens, 100 * (1 - MAX_POS_RATE))
        best_idx = np.searchsorted(p_sorted[::-1], cutoff, side='right')
        best_idx = len(p_sorted) - 1 - best_idx
        best_idx = np.clip(best_idx, 0, len(p_sorted) - 1)

optimal_threshold = float(thr_k[best_idx])
val_precision     = float(prec_k[best_idx])
val_recall        = float(rec_k[best_idx])
val_pos_rate      = float(pos_rate_k[best_idx])

logp(f"Umbral final: {optimal_threshold:.6f}")
logp(f"VAL → P={val_precision:.3f} | R={val_recall:.3f} | pos_rate={val_pos_rate*100:.1f}%")

# ==========================
# 5) MÉTRICAS EN TEST
# ==========================
logp("Evaluación en TEST…")

y_pred_test = (proba_test_ens >= optimal_threshold).astype(int)

test_precision = precision_score(y_test, y_pred_test)
test_recall    = recall_score(y_test, y_pred_test)
test_f1        = f1_score(y_test, y_pred_test)
test_f05       = fbeta_score(y_test, y_pred_test, beta=0.5)
test_pr_auc    = average_precision_score(y_test, proba_test_ens)
test_roc_auc   = roc_auc_score(y_test, proba_test_ens)

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = map(int, cm.ravel())

print("\n" + "="*80)
print("📊 CLASSIFICATION REPORT (TEST)")
print("="*80)
print(classification_report(y_test, y_pred_test, target_names=['No Accidente','Accidente']))

print(f"\n🎯 MÉTRICAS (TEST):")
print(f"   Precision:  {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"   Recall:     {test_recall:.4f} ({test_recall*100:.2f}%)")
print(f"   F1-Score:   {test_f1:.4f}")
print(f"   F0.5-Score: {test_f05:.4f}")
print(f"   PR-AUC:     {test_pr_auc:.4f}")
print(f"   ROC-AUC:    {test_roc_auc:.4f}")

print(f"\n🔢 CONFUSIÓN:")
print(f"   TN: {tn:,} | FP: {fp:,}")
print(f"   FN: {fn:,} | TP: {tp:,}")

print(f"\n🎯 OBJETIVOS:")
print(f"   Recall ≥ 70%: {'✅' if test_recall >= 0.70 else '⚠️'}  (Actual: {test_recall*100:.1f}%)")
print(f"   Precision ≥ 5%: {'✅' if test_precision >= 0.05 else '⚠️'} (Actual: {test_precision*100:.1f}%)")

# ==========================
# 6) GUARDAR PREDICCIONES CSV (VAL & TEST)
# ==========================
logp("Guardando predicciones a CSV…")

val_df = pd.DataFrame({
    'row_idx': np.arange(len(y_val)),
    'y_true':  y_val.astype(int),
    'proba':   proba_val_ens.astype(float),
    'y_pred':  (proba_val_ens >= optimal_threshold).astype(int)
})
val_df['threshold_used'] = optimal_threshold
val_df['split'] = 'val'
val_df['risk_band'] = pd.cut(val_df['proba'], bins=[-np.inf, 0.05, 0.10, np.inf],
                             labels=['low','medium','high'])

test_df = pd.DataFrame({
    'row_idx': np.arange(len(y_test)),
    'y_true':  y_test.astype(int),
    'proba':   proba_test_ens.astype(float),
    'y_pred':  y_pred_test.astype(int)
})
test_df['threshold_used'] = optimal_threshold
test_df['split'] = 'test'
test_df['risk_band'] = pd.cut(test_df['proba'], bins=[-np.inf, 0.05, 0.10, np.inf],
                              labels=['low','medium','high'])

preds_val_path  = os.path.join(RESULTS_DIR, "predictions_val_ensemble.csv")
preds_test_path = os.path.join(RESULTS_DIR, "predictions_test_ensemble.csv")
val_df.to_csv(preds_val_path, index=False)
test_df.to_csv(preds_test_path, index=False)

logp(f"Predicciones VAL:  {preds_val_path}")
logp(f"Predicciones TEST: {preds_test_path}")

# ==========================
# 7) HISTÓRICOS COMPLETOS (opcional)
# ==========================
if df_original is not None:
    logp("Extrayendo históricos completos…")
    accidents_df = df_original[df_original['accident'] == 1].copy()
    historical_data_full = {
        'total_accidents': int(len(accidents_df)),
        'years_covered': sorted(accidents_df['year'].unique().tolist()),
        'accidents_by_year': accidents_df.groupby('year').size().to_dict(),
        'accidents_by_month': accidents_df.groupby('month').size().to_dict(),
        'accidents_by_hour': accidents_df.groupby('hour').size().to_dict(),
        'accidents_by_dow': accidents_df.groupby('day_of_week').size().to_dict(),
        'accidents_by_cluster': accidents_df.groupby('cluster_id').size().to_dict(),
        'accidents_by_year_month': accidents_df.groupby(['year','month']).size().to_dict(),
        'accidents_by_hour_dow': accidents_df.groupby(['hour','day_of_week']).size().to_dict(),
        'peak_hours': accidents_df.groupby('hour').size().nlargest(5).to_dict(),
        'safest_hours': accidents_df.groupby('hour').size().nsmallest(5).to_dict(),
        'most_dangerous_clusters': accidents_df.groupby('cluster_id').size().nlargest(10).to_dict(),
        'safest_clusters': accidents_df.groupby('cluster_id').size().nsmallest(10).to_dict()
    }
else:
    historical_data_full = {
        'total_accidents': int(y_train.sum() + y_val.sum() + y_test.sum()),
        'note': 'Dataset original no disponible'
    }

# ==========================
# 8) GUARDAR MODELOS
# ==========================
logp("Guardando modelos y artefactos…")

# Guardamos calibradores del ensemble y el umbral óptimo
streamlit_model = {
    "ensemble": {
        "calibrators": calibrators,  # cada uno lleva el estimator interno calibrado
        "n_models": len(calibrators),
        "optimal_threshold": float(optimal_threshold)
    },
    "scaler": scaler,
    "selector": selector,
    "selected_features": selected_features,
    "feature_cols": feature_cols,
    "cluster_geometries": cluster_geometries,
    "historical_data": historical_data_full,

    "performance": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1),
        "f05_score": float(test_f05),
        "pr_auc": float(test_pr_auc),
        "roc_auc": float(test_roc_auc),
        "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}
    },
    "config": {
        "target_recall": TARGET_RECALL,
        "min_precision": MIN_PRECISION,
        "max_pos_rate_val": MAX_POS_RATE,
        "created_at": datetime.now().isoformat(),
        "model_version": "3.0-ensemble-calib-valid"
    }
}

# .joblib completo
joblib_path = os.path.join(OUTPUT_DIR, "barcelona_accident_model_ensemble_2.joblib")
joblib.dump(streamlit_model, joblib_path, compress=3)

# .pkl mínimo (solo para inferencia rápida si lo necesitases)
minimal_model = {
    "ensemble": streamlit_model["ensemble"],
    "scaler": scaler,
    "selector": selector,
    "selected_features": selected_features,
    "feature_cols": feature_cols,
    "performance": streamlit_model["performance"]
}
pkl_path = os.path.join(OUTPUT_DIR, "barcelona_accident_model_ensemble_2.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(minimal_model, f)

# ==========================
# 9) REPORTE FINAL
# ==========================
total_time = (time.time() - start_time) / 60
print("\n" + "="*80)
print("✅ SCRIPT 2 PRO COMPLETADO")
print("="*80)
print(f"📊 TEST → P={test_precision:.4f}  R={test_recall:.4f}  F1={test_f1:.4f}  PR-AUC={test_pr_auc:.4f}")
print(f"💾 Guardados:")
print(f"   Modelo .joblib: {joblib_path}")
print(f"   Modelo .pkl:    {pkl_path}")
print(f"   Preds VAL CSV:  {preds_val_path}")
print(f"   Preds TEST CSV: {preds_test_path}")
print(f"⏱️ Tiempo total: {total_time:.1f} min")

# Limpieza
del X_train, y_train, X_val, y_val, X_test, y_test
if df_original is not None:
    del df_original
gc.collect()
logp("🎉 Listo para Streamlit.")


🚀 SCRIPT 2 PRO: ENSEMBLE + CALIBRACIÓN VALID + RECALL-FIRST
[10:59:26] Cargando datos procesados...
[10:59:27] Tamaños -> Train: 704,254 | Val: 442,951 | Test: 447,612
[10:59:27] Tasa accidente -> Train: 7.41% | Val: 1.74%
[10:59:27] Cargando dataset original para históricos…
[10:59:30] Original: 3,495,160 filas
[10:59:30] RandomizedSearch ampliado (scoring=average_precision)…
Fitting 3 folds for each of 48 candidates, totalling 144 fits
[13:56:45] ✅ Mejor AP (CV): 0.3376
[13:56:45] Entrenando ensemble top-3 + calibración en validación…
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[22]	valid_0's binary_logloss: 0.637965
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[700]	valid_0's binary_logloss: 0.495954
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[900]	valid_0's binary_logloss: 0.497993
[14:01:02] Buscando u